# Create Customers data

Starting with three csv files related to customer's info, my goal is to perform data transformation before combining into one file.

In [1]:
import duckdb

conn = duckdb.connect()

%load_ext sql
%sql conn --alias duckdb
%config SqlMagic.displaylimit = None

displaylimit: Value None will be treated as 0 (no limit)

In [2]:
%sql CREATE TABLE IF NOT EXISTS cust_info AS SELECT * FROM 'datasets/source_crm/cust_info.csv'
%sql CREATE TABLE IF NOT EXISTS CUST_AZ12 AS SELECT * FROM 'datasets/source_erp/CUST_AZ12.csv'
%sql CREATE TABLE IF NOT EXISTS LOC_A101 AS SELECT * FROM 'datasets/source_erp/LOC_A101.csv'

Running query in 'duckdb'

Running query in 'duckdb'

Running query in 'duckdb'

Count
18484


In [3]:
%%sql
-- Check if all three files are loaded.
    
SELECT table_name
FROM information_schema.tables

Running query in 'duckdb'

table_name
CUST_AZ12
cust_info
LOC_A101


In [4]:
%%sql
-- Check the structures of the files
    
SELECT table_name, column_name, data_type
FROM information_schema.columns

Running query in 'duckdb'

table_name,column_name,data_type
CUST_AZ12,CID,VARCHAR
CUST_AZ12,BDATE,DATE
CUST_AZ12,GEN,VARCHAR
cust_info,cst_id,BIGINT
cust_info,cst_key,VARCHAR
cust_info,cst_firstname,VARCHAR
cust_info,cst_lastname,VARCHAR
cust_info,cst_marital_status,VARCHAR
cust_info,cst_gndr,VARCHAR
cust_info,cst_create_date,DATE


I must ensure that all ID columns from each table are unique so they can form relations between each tables. 

## Customer info

In [5]:
%%sql

SELECT *
FROM cust_info
LIMIT 5;

Running query in 'duckdb'

cst_id,cst_key,cst_firstname,cst_lastname,cst_marital_status,cst_gndr,cst_create_date
11000,AW00011000,Jon,Yang,M,M,2025-10-06
11001,AW00011001,Eugene,Huang,S,M,2025-10-06
11002,AW00011002,Ruben,Torres,M,M,2025-10-06
11003,AW00011003,Christy,Zhu,S,F,2025-10-06
11004,AW00011004,Elizabeth,Johnson,S,F,2025-10-06


In [6]:
%%sql
-- Check for empty customer id.
    
SELECT *
FROM cust_info
WHERE cst_id IS NULL;

Running query in 'duckdb'

cst_id,cst_key,cst_firstname,cst_lastname,cst_marital_status,cst_gndr,cst_create_date
None,SF566,None,None,None,None,None
None,PO25,None,None,None,None,None
None,13451235,None,None,None,None,None
None,A01Ass,None,None,None,None,None


In [7]:
%%sql
-- Use group by to check duplicate id
    
SELECT cst_id, COUNT(cst_id)
FROM cust_info
GROUP BY cst_id
ORDER BY COUNT(cst_id) DESC
LIMIT 10;

Running query in 'duckdb'

cst_id,count(cst_id)
29466,3
29433,2
29449,2
29473,2
29483,2
11004,1
11028,1
11010,1
11073,1
11078,1


In [8]:
%%sql
-- Analyze duplicate id
    
SELECT *
FROM cust_info
WHERE cst_id IN (29466, 29483, 29433, 29473, 29449);

Running query in 'duckdb'

cst_id,cst_key,cst_firstname,cst_lastname,cst_marital_status,cst_gndr,cst_create_date
29433,AW00029433,None,None,M,M,2026-01-25
29433,AW00029433,Thomas,King,M,M,2026-01-27
29449,AW00029449,None,Chen,S,None,2026-01-25
29449,AW00029449,Laura,Chen,S,F,2026-01-26
29466,AW00029466,None,None,None,None,2026-01-25
29466,AW00029466,Lance,Jimenez,M,None,2026-01-26
29466,AW00029466,Lance,Jimenez,M,M,2026-01-27
29473,AW00029473,Carmen,None,None,None,2026-01-25
29473,AW00029473,Carmen,Subram,S,None,2026-01-26
29483,AW00029483,None,Navarro,None,None,2026-01-25


In [9]:
%%sql
-- Delete null cst_id values

DELETE FROM cust_info
WHERE cst_id IS NULL;

Running query in 'duckdb'

Count
4


In [10]:
%%sql
-- Delete 29433 id duplicate

DELETE FROM cust_info
WHERE cst_id = 29433 AND cst_firstname IS NULL;

Running query in 'duckdb'

Count
1


In [11]:
%%sql
-- Delete 29449	id duplicate

DELETE FROM cust_info
WHERE cst_id = 29449 AND cst_firstname IS NULL;

Running query in 'duckdb'

Count
1


In [12]:
%%sql
-- Delete 29466	id duplicate

DELETE FROM cust_info
WHERE cst_id = 29466 AND cst_gndr IS NULL;

Running query in 'duckdb'

Count
2


In [13]:
%%sql
-- Delete 29473	id duplicate

DELETE FROM cust_info
WHERE cst_id = 29473 AND cst_lastname IS NULL;

Running query in 'duckdb'

Count
1


In [14]:
%%sql
-- Delete 29483	id duplicate

DELETE FROM cust_info
WHERE cst_id = 29483 AND cst_firstname IS NULL;

Running query in 'duckdb'

Count
1


In [15]:
%%sql
-- Check for duplicate values again
    
SELECT COUNT(*), COUNT(DISTINCT cst_id)
FROM cust_info;

Running query in 'duckdb'

count_star(),count(DISTINCT cst_id)
18484,18484


With data cleaning completed, I create a new table to be exported later.

In [16]:
%%sql

CREATE TABLE IF NOT EXISTS cust_info_new AS
SELECT 
    cst_id, 
    cst_key,
    TRIM(cst_firstname) AS cst_firstname, 
    TRIM(cst_lastname) AS cst_lastname, 
    CASE
        WHEN cst_marital_status = 'M' THEN 'Married'
        WHEN cst_marital_status = 'S' THEN 'Single'
        ELSE 'N/A'
    END AS cst_marital_status, 
    CASE
        WHEN cst_gndr = 'M' THEN 'Male'
        WHEN cst_gndr = 'F' THEN 'Female'
        ELSE 'N/A'
    END AS cst_gndr,
    cst_create_date
FROM cust_info;

Running query in 'duckdb'

Count
18484


## Customer birthdate

In [17]:
%%sql

SELECT *
FROM CUST_AZ12
LIMIT 5;

Running query in 'duckdb'

CID,BDATE,GEN
NASAW00011000,1971-10-06,Male
NASAW00011001,1976-05-10,Male
NASAW00011002,1971-02-09,Male
NASAW00011003,1973-08-14,Female
NASAW00011004,1979-08-05,Female


In [18]:
%%sql
-- Removes future birthdates
    
SELECT
    CASE
        WHEN BDATE > today() THEN NULL
        ELSE BDATE
    END AS BDATE
FROM CUST_AZ12
ORDER BY BDATE DESC
LIMIT 10;

Running query in 'duckdb'

BDATE
1986-06-25
1986-06-25
1986-06-23
1986-06-23
1986-06-22
1986-06-22
1986-06-05
1986-06-05
1986-06-05
1986-06-05


In [19]:
%%sql
-- Check for duplicate ID
    
SELECT COUNT(*), COUNT(DISTINCT CID)
FROM CUST_AZ12;

Running query in 'duckdb'

count_star(),count(DISTINCT CID)
18484,18484


In [20]:
%%sql
-- Only include relevant texts in ID column
    
SELECT RIGHT(CID, 10) AS CID, BDATE, GEN
FROM CUST_AZ12
LIMIT 5;

Running query in 'duckdb'

CID,BDATE,GEN
AW00011000,1971-10-06,Male
AW00011001,1976-05-10,Male
AW00011002,1971-02-09,Male
AW00011003,1973-08-14,Female
AW00011004,1979-08-05,Female


In [21]:
%%sql
-- Check gender column
    
SELECT DISTINCT GEN
FROM CUST_AZ12;

Running query in 'duckdb'

GEN
""
None
M
""
Female
F
Male
M
F


In [22]:
%%sql
-- Replace F and M text to be female and male and remove others as null values.
    
SELECT
    CASE
        WHEN TRIM(GEN) IN ('F', 'Female') THEN 'Female'
        WHEN TRIM(GEN) IN ('M', 'Male') THEN 'Male'
        ELSE 'N/A'
    END AS GEN
FROM CUST_AZ12
LIMIT 5;

Running query in 'duckdb'

GEN
Male
Male
Male
Female
Female


With the final step done, I create another clean table.

In [23]:
%%sql

CREATE TABLE IF NOT EXISTS cust_bdate AS
SELECT 
    RIGHT(CID, 10) AS CID, 
    CASE
        WHEN BDATE > today() THEN NULL
        ELSE BDATE
    END AS BDATE, 
    CASE
        WHEN TRIM(GEN) IN ('F', 'Female') THEN 'Female'
        WHEN TRIM(GEN) IN ('M', 'Male') THEN 'Male'
        ELSE 'N/A'
    END AS GEN
FROM CUST_AZ12;

Running query in 'duckdb'

Count
18484


## Customer Location

In [24]:
%%sql

SELECT *
FROM LOC_A101
LIMIT 5;

Running query in 'duckdb'

CID,CNTRY
AW-00011000,Australia
AW-00011001,Australia
AW-00011002,Australia
AW-00011003,Australia
AW-00011004,Australia


In [25]:
%%sql
-- Check for duplicate ID
    
SELECT COUNT(*), COUNT(DISTINCT CID)
FROM LOC_A101;

Running query in 'duckdb'

count_star(),count(DISTINCT CID)
18484,18484


In [26]:
%%sql
-- Remove '-' from ID column
    
SELECT REPLACE(CID, '-', ''), CNTRY
FROM LOC_A101
LIMIT 5;

Running query in 'duckdb'

"""replace""(CID, '-', '')",CNTRY
AW00011000,Australia
AW00011001,Australia
AW00011002,Australia
AW00011003,Australia
AW00011004,Australia


In [27]:
%%sql
--Check country column
    
SELECT DISTINCT TRIM(CNTRY)
FROM LOC_A101;

Running query in 'duckdb'

"main.""trim""(CNTRY)"
United Kingdom
Canada
France
USA
Germany
None
""
United States
Australia
DE


In [28]:
%%sql
-- Ensure country texts are consistent.
    
SELECT
    CASE 
        WHEN TRIM(CNTRY) IN ('USA', 'US', 'United States') THEN 'United States'
        WHEN TRIM(CNTRY) IN ('Germany', 'DE') THEN 'Germany'
        WHEN TRIM(cntry) = '' OR cntry IS NULL THEN 'N/A'
        ELSE TRIM(CNTRY)
    END AS CNTRY
FROM LOC_A101
LIMIT 5;

Running query in 'duckdb'

CNTRY
Australia
Australia
Australia
Australia
Australia


Now create a final clean table.

In [29]:
%%sql

CREATE TABLE IF NOT EXISTS cust_loc AS
SELECT 
    REPLACE(CID, '-', '') AS CID, 
    CASE 
        WHEN TRIM(CNTRY) IN ('USA', 'US', 'United States') THEN 'United States'
        WHEN TRIM(CNTRY) IN ('Germany', 'DE') THEN 'Germany'
        WHEN TRIM(cntry) = '' OR cntry IS NULL THEN 'N/A'
        ELSE TRIM(CNTRY)
    END AS CNTRY
FROM LOC_A101

Running query in 'duckdb'

Count
18484


## Join tables

With all three clean tables created, I can join tables to create a customer information table for easier analysis.

In [30]:
%%sql

SELECT table_name
FROM information_schema.tables

Running query in 'duckdb'

table_name
CUST_AZ12
cust_bdate
cust_info
cust_info_new
cust_loc
LOC_A101


In [31]:
%%sql
-- Join tables based on their ID columns.

SELECT cst_id, cst_key, cst_firstname, cst_lastname, cst_marital_status, COALESCE(cst_gndr, GEN, 'N/A') AS GEN, cst_create_date, BDATE, CNTRY
FROM cust_info_new
JOIN cust_bdate
ON cust_info_new.cst_key = cust_bdate.CID
JOIN cust_loc
ON cust_info_new.cst_key = cust_loc.CID
LIMIT 5;

Running query in 'duckdb'

cst_id,cst_key,cst_firstname,cst_lastname,cst_marital_status,GEN,cst_create_date,BDATE,CNTRY
11000,AW00011000,Jon,Yang,Married,Male,2025-10-06,1971-10-06,Australia
11001,AW00011001,Eugene,Huang,Single,Male,2025-10-06,1976-05-10,Australia
11002,AW00011002,Ruben,Torres,Married,Male,2025-10-06,1971-02-09,Australia
11003,AW00011003,Christy,Zhu,Single,Female,2025-10-06,1973-08-14,Australia
11004,AW00011004,Elizabeth,Johnson,Single,Female,2025-10-06,1979-08-05,Australia


In [32]:
%%sql
-- Now create a complete table and export as csv file.
    
COPY (
    SELECT 
        cst_id AS customer_id, 
        cst_key AS customer_key, 
        cst_firstname AS first_name, 
        cst_lastname AS last_name, 
        cst_marital_status AS marital_status, 
        COALESCE(cst_gndr, GEN, 'N/A') AS gender, 
        cst_create_date AS create_date, 
        BDATE AS birthdate, 
        CNTRY AS country
    FROM cust_info_new
    JOIN cust_bdate
    ON cust_info_new.cst_key = cust_bdate.CID
    JOIN cust_loc
    ON cust_info_new.cst_key = cust_loc.CID
) 
TO 'datasets/customers_info.csv' (HEADER, DELIMITER ',');

Running query in 'duckdb'

Count
18484


In [33]:
# Done

conn.close()